In [11]:
# Import the libraries needed for data handling, reproducibility, and dataset splitting


import pandas as pd
import numpy as np
import sklearn

from pathlib import Path
from sklearn.model_selection import train_test_split

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 3.0.3
NumPy version: 2.4.6
Scikit-learn version: 1.9.0


In [12]:
# Define the file path for the cleaned V3 CICIDS2017 dataset
# and verify that Jupyter can locate the file

data_file = Path(
    "/Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/cicids2017_cleaned_WHOLE_SET_V3.csv"
)

print("File exists:", data_file.exists())
print("File:", data_file)

File exists: True
File: /Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/cicids2017_cleaned_WHOLE_SET_V3.csv


In [3]:
# Load the complete cleaned V3 dataset into a Pandas DataFrame
# and display its total number of rows and columns

df = pd.read_csv(data_file)

print("Dataset loaded.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded.
Rows: 2519196
Columns: 53


In [4]:
# Define the target classification column
# and display the number of records belonging to each attack category

label_col = "Attack Type"

print(df[label_col].value_counts())

Attack Type
Normal Traffic    2094199
DoS                193615
DDoS               128011
Port Scanning       90130
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64


In [5]:
# Create the first stratified split:
# 85% of the dataset is temporarily kept for training/validation
# and 15% is reserved as the final test set.
# Stratification preserves the class distribution across the split.
# random_state=42 makes the split reproducible.

train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df[label_col]
)

print("Train + Validation:", len(train_val_df))
print("Reserved Test:", len(test_df))

Train + Validation: 2141316
Reserved Test: 377880


In [6]:
# Split the remaining 85% into training and validation sets.
# Using 15/85 as the validation fraction produces an overall
# dataset split of approximately 70% training, 15% validation, and 15% test.

validation_fraction = 15 / 85

train_df, val_df = train_test_split(
    train_val_df,
    test_size=validation_fraction,
    random_state=42,
    stratify=train_val_df[label_col]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Reserved Test:", len(test_df))

Training: 1763436
Validation: 377880
Reserved Test: 377880


In [7]:
# Verify that the three partitions contain approximately
# 70% training, 15% validation, and 15% reserved testing data.
# Also verify that no records were lost during splitting.

total = len(df)

print(f"Training:   {len(train_df):,} ({len(train_df)/total*100:.2f}%)")
print(f"Validation: {len(val_df):,} ({len(val_df)/total*100:.2f}%)")
print(f"Test:       {len(test_df):,} ({len(test_df)/total*100:.2f}%)")

print("\nTotal after splitting:",
      len(train_df) + len(val_df) + len(test_df))

print("Original total:", total)

Training:   1,763,436 (70.00%)
Validation: 377,880 (15.00%)
Test:       377,880 (15.00%)

Total after splitting: 2519196
Original total: 2519196


In [8]:
# Compare the class percentages in the full dataset and each partition
# to verify that stratification preserved the original attack distribution.

def class_percentages(data):
    return (
        data[label_col]
        .value_counts(normalize=True)
        .mul(100)
        .sort_index()
    )

distribution_check = pd.DataFrame({
    "Whole Dataset": class_percentages(df),
    "Training": class_percentages(train_df),
    "Validation": class_percentages(val_df),
    "Test": class_percentages(test_df)
})

distribution_check

,Whole Dataset,Training,Validation,Test
Attack Type,,,,
Bots,0.077326,0.077349,0.077273,0.077273
Brute Force,0.363211,0.363211,0.363078,0.363343
DDoS,5.081423,5.081387,5.081507,5.081507
DoS,7.685587,7.685621,7.685509,7.685509
Normal Traffic,83.129657,83.129697,83.129565,83.129565
Port Scanning,3.577729,3.577674,3.577855,3.577855
Web Attacks,0.085067,0.085061,0.085212,0.084948


In [9]:
# Generate a hash for every complete record in each partition.
# These hashes allow us to check whether identical records
# accidentally appear in more than one dataset partition.

train_hash = pd.util.hash_pandas_object(
    train_df,
    index=False
)

val_hash = pd.util.hash_pandas_object(
    val_df,
    index=False
)

test_hash = pd.util.hash_pandas_object(
    test_df,
    index=False
)

train_hash_set = set(train_hash)
val_hash_set = set(val_hash)
test_hash_set = set(test_hash)

print(
    "Train ↔ Validation overlap:",
    len(train_hash_set & val_hash_set)
)

print(
    "Train ↔ Test overlap:",
    len(train_hash_set & test_hash_set)
)

print(
    "Validation ↔ Test overlap:",
    len(val_hash_set & test_hash_set)
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0


In [13]:
# Verify that identical 52-feature vectors do not appear across partitions,
# even if their Attack Type labels were somehow different.
# This is a stricter check than comparing complete rows.

feature_cols = [
    column for column in df.columns
    if column != label_col
]

train_feature_hash = pd.util.hash_pandas_object(
    train_df[feature_cols],
    index=False
)

val_feature_hash = pd.util.hash_pandas_object(
    val_df[feature_cols],
    index=False
)

test_feature_hash = pd.util.hash_pandas_object(
    test_df[feature_cols],
    index=False
)

train_feature_set = set(train_feature_hash)
val_feature_set = set(val_feature_hash)
test_feature_set = set(test_feature_hash)

print(
    "Train ↔ Validation feature overlap:",
    len(train_feature_set & val_feature_set)
)

print(
    "Train ↔ Test feature overlap:",
    len(train_feature_set & test_feature_set)
)

print(
    "Validation ↔ Test feature overlap:",
    len(val_feature_set & test_feature_set)
)

Train ↔ Validation feature overlap: 0
Train ↔ Test feature overlap: 0
Validation ↔ Test feature overlap: 0


In [14]:
# Create a separate folder to store the new V3
# training, validation, and reserved test partitions.

split_folder = data_file.parent / "V3_Stratified_Split"

split_folder.mkdir(
    exist_ok=True
)

print("Split files will be saved to:")
print(split_folder)

Split files will be saved to:
/Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/V3_Stratified_Split


In [15]:
# Define descriptive filenames for the three new dataset partitions.
# The word RESERVED is included in the test filename to remind the team
# that it should not be used for model training or tuning.

train_file = (
    split_folder /
    "cicids2017_V3_train_70.csv"
)

val_file = (
    split_folder /
    "cicids2017_V3_validation_15.csv"
)

test_file = (
    split_folder /
    "cicids2017_V3_test_RESERVED_15.csv"
)

print("Training file:", train_file.name)
print("Validation file:", val_file.name)
print("Reserved test file:", test_file.name)

Training file: cicids2017_V3_train_70.csv
Validation file: cicids2017_V3_validation_15.csv
Reserved test file: cicids2017_V3_test_RESERVED_15.csv


In [16]:
# Save the training, validation, and reserved test partitions
# as separate CSV files without adding Pandas index numbers.

train_df.to_csv(
    train_file,
    index=False
)

val_df.to_csv(
    val_file,
    index=False
)

test_df.to_csv(
    test_file,
    index=False
)

print("Training saved to:", train_file)
print("Validation saved to:", val_file)
print("Reserved test saved to:", test_file)

Training saved to: /Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/V3_Stratified_Split/cicids2017_V3_train_70.csv
Validation saved to: /Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/V3_Stratified_Split/cicids2017_V3_validation_15.csv
Reserved test saved to: /Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/V3_Stratified_Split/cicids2017_V3_test_RESERVED_15.csv


In [17]:
# Confirm that all three split files were successfully created on disk.

print("Training file exists:", train_file.exists())
print("Validation file exists:", val_file.exists())
print("Reserved test file exists:", test_file.exists())

Training file exists: True
Validation file exists: True
Reserved test file exists: True
